# 08 · XGBoost Deep Dive & Fine-Tuning — *from scratch*

XGBoost is one of the strongest models for tabular data. This self-contained
notebook covers three things a beginner needs:

1. **Serving it safely** — a famous version-compatibility gotcha with
   scikit-learn `Pipeline`s, and the robust way around it.
2. **A fair head-to-head** with LightGBM on the held-out test set.
3. **Fine-tuning** — *the* part most tutorials skip: what every important
   hyperparameter does and exactly how to search for good values.

Everything below imports only public libraries — nothing from the project.

In [1]:
# --- Self-contained setup (schema, cleaning, preprocessor, bands) ----------
import warnings
warnings.filterwarnings("ignore")   # hide library deprecation notices for clean output

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

DATA_PATH     = "../data/raw/cars24-car-price-cleaned-new.csv.gz"
TARGET        = "selling_price"
TARGET_ENCODE = ["make", "model"]
NUMERIC       = ["km_driven", "mileage", "engine", "max_power", "age"]
FLAGS         = ["Individual", "Trustmark Dealer", "Diesel", "Electric", "LPG",
                 "Petrol", "Manual", "Seats_5", "Seats_Above_5"]
FEATURES      = TARGET_ENCODE + NUMERIC + FLAGS
BAND_LABELS   = ["Low", "Medium", "High"]
TEST_SIZE, RANDOM_STATE = 0.2, 42


def clean(df):
    df = df.rename(columns={"5": "Seats_5", ">5": "Seats_Above_5"})
    df = df.dropna(subset=[TARGET])
    df = df[df[TARGET] > 0].dropna()
    for c in TARGET_ENCODE:
        df[c] = df[c].astype(str).str.strip().str.upper()
    return df.reset_index(drop=True)


def build_preprocessor():
    return ColumnTransformer([
        ("target_enc", TargetEncoder(target_type="continuous", random_state=RANDOM_STATE), TARGET_ENCODE),
        ("passthrough", "passthrough", NUMERIC + FLAGS),
    ], remainder="drop")


def band_edges(prices):
    edges = [float(np.quantile(prices, q)) for q in (0.0, 1/3, 2/3, 1.0)]
    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = edges[i - 1] + 0.01
    return [round(e, 2) for e in edges]


def price_to_band(prices, edges):
    prices = np.asarray(prices, dtype=float)
    idx = np.clip(np.digitize(prices, edges[1:-1]), 0, len(BAND_LABELS) - 1)
    return np.array(BAND_LABELS)[idx]


# The baseline XGBoost settings this project uses (modest + comparable).
XGB_BASE_PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.05,
                       subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                       random_state=RANDOM_STATE, verbosity=0)

df = clean(pd.read_csv(DATA_PATH))
X, y = df[FEATURES].copy(), df[TARGET].copy()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
edges = band_edges(ytr)
print("train/test:", len(Xtr), "/", len(Xte))

train/test: 15856 / 3964


## 1 · A serving gotcha: XGBoost inside a scikit-learn `Pipeline`
On **older** stacks (`xgboost 2.1` × `scikit-learn 1.6`) an XGBoost model *fits*
inside a `Pipeline` but throws when you call **`.predict()`** — a mismatch in how
the two libraries negotiate "estimator tags". On newer versions it's fixed. The
cell below simply *checks* which world you're in, so the notebook is honest on
any install.

In [2]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([("preprocess", build_preprocessor()),
                 ("model", XGBRegressor(**XGB_BASE_PARAMS))])
pipe.fit(Xtr, ytr)                       # fitting always works
try:
    pipe.predict(Xte[:3])
    print("predict OK  -> your xgboost x scikit-learn versions are compatible")
except Exception as e:
    print("predict FAILS ->", type(e).__name__ + ":", str(e)[:80])

import xgboost, sklearn
print("xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__)

predict OK  -> your xgboost x scikit-learn versions are compatible
xgboost 3.3.0 | scikit-learn 1.9.0


### The robust fix: **decoupled** serving
Whatever your versions, this always works: fit the **preprocessor** and the
**regressor** as two separate objects, and predict on `pre.transform(X)`. It
sidesteps the `Pipeline` tag negotiation entirely — and it's how we evaluate
XGBoost for the rest of this notebook.

In [3]:
# Encode once with a fitted preprocessor, then work with the numeric arrays.
pre = build_preprocessor()
Xtr_e = pre.fit_transform(Xtr, ytr)
Xte_e = pre.transform(Xte)

xgb = XGBRegressor(**XGB_BASE_PARAMS)
xgb.fit(Xtr_e, ytr)
pred = xgb.predict(Xte_e)


def score(y_true, y_pred):
    """R2 / MAE / RMSE plus the derived Low-Medium-High band accuracy."""
    tb = price_to_band(y_true, edges)
    pb = price_to_band(y_pred, edges)
    return {"R2":   round(r2_score(y_true, y_pred), 4),
            "MAE":  round(mean_absolute_error(y_true, y_pred), 3),
            "RMSE": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
            "band_acc": round(float((tb == pb).mean()), 4)}


print("XGBoost (decoupled) TEST:", score(yte, pred))

XGBoost (decoupled) TEST: {'R2': 0.9574, 'MAE': 0.665, 'RMSE': 1.014, 'band_acc': 0.8613}


## 2 · Head-to-head vs LightGBM
Is XGBoost actually better than LightGBM here? Let's train both the same way and
compare on the held-out test set. (If LightGBM isn't installed, we skip it.)

In [4]:
rows = {"XGBoost": score(yte, pred)}
try:
    from lightgbm import LGBMRegressor
    lgbm = LGBMRegressor(n_estimators=600, num_leaves=63, learning_rate=0.05,
                         subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                         random_state=RANDOM_STATE, verbosity=-1)
    lgbm.fit(Xtr_e, ytr)
    rows["LightGBM"] = score(yte, lgbm.predict(Xte_e))
except ImportError:
    print("lightgbm not installed — showing XGBoost only")

pd.DataFrame(rows).T

,R2,MAE,RMSE,band_acc
XGBoost,0.9574,0.665,1.014,0.8613
LightGBM,0.9568,0.664,1.022,0.8582


**Typical result:** the two are *statistically indistinguishable* — identical R²,
MAE within a few hundred rupees. That's the real-world lesson: once you're at the
top of the leaderboard, the choice between XGBoost and LightGBM is usually about
**operational fit** (serialisation, serving, speed), not accuracy.

## 3 · Fine-tuning XGBoost — the knobs that matter
XGBoost has many hyperparameters, but they fall into **four intuitive groups**.
You rarely touch more than these.

| Group | Hyperparameter | What it controls | Turning it **up** → |
| :-- | :-- | :-- | :-- |
| **Learning schedule** | `learning_rate` (η) | How strongly each new tree corrects the last | Faster but rougher; can overshoot |
| | `n_estimators` | Number of boosting rounds (trees) | More capacity; slower; can overfit |
| **Tree complexity** | `max_depth` | How deep each tree grows = interaction order | Richer interactions; overfits |
| | `min_child_weight` | Min. data weight required in a leaf | **Simpler** trees (regularises) |
| | `gamma` | Min. loss reduction needed to split | **Simpler** trees (regularises) |
| **Randomness** | `subsample` | Fraction of **rows** used per tree | Less overfitting; too low underfits |
| | `colsample_bytree` | Fraction of **columns** used per tree | Decorrelates trees; too low underfits |
| **Regularisation** | `reg_lambda` (L2) | Penalty on leaf-weight size | Smoother, more conservative |
| | `reg_alpha` (L1) | Sparsity penalty on leaf weights | Pushes some weights to zero |

**The single most important relationship:** `learning_rate` and `n_estimators`
trade off. A *smaller* learning rate needs *more* trees but usually generalises
better. So the standard recipe is: pick a small learning rate, then let
**early stopping** choose `n_estimators` automatically.

In [5]:
# The baseline we're starting from:
for k in ["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree"]:
    print(f"{k:18s} {XGB_BASE_PARAMS[k]}")

n_estimators       500
max_depth          6
learning_rate      0.05
subsample          0.9
colsample_bytree   0.9


### Tune by **searching**, not guessing
`RandomizedSearchCV` samples random combinations from ranges you specify and
scores each with cross-validation (so it's leakage-safe). It explores a big space
far more efficiently than trying every combination on a grid. The demo below is
deliberately small (`n_iter=15`) so it finishes quickly — widen the ranges and
raise `n_iter` for a serious search.

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# uniform(loc, scale) samples in [loc, loc + scale]
search_space = {
    "n_estimators":     randint(300, 900),    # boosting rounds
    "max_depth":        randint(3, 9),        # interaction depth
    "learning_rate":    uniform(0.02, 0.10),  # shrinkage 0.02 .. 0.12
    "subsample":        uniform(0.70, 0.30),  # rows    0.70 .. 1.00
    "colsample_bytree": uniform(0.70, 0.30),  # columns 0.70 .. 1.00
    "min_child_weight": randint(1, 8),        # larger => simpler trees
    "reg_lambda":       uniform(0.0, 3.0),    # L2 penalty
}

base = XGBRegressor(objective="reg:squarederror", tree_method="hist",
                    n_jobs=-1, random_state=RANDOM_STATE, verbosity=0)

search = RandomizedSearchCV(
    base, search_space, n_iter=15, cv=3,
    scoring="neg_mean_absolute_error",     # CV MAE in Lakhs
    random_state=RANDOM_STATE, n_jobs=-1,
)
search.fit(Xtr_e, ytr)

print("best CV MAE :", round(-search.best_score_, 3), "Lakhs")
print("best params :")
for k, v in sorted(search.best_params_.items()):
    print(f"   {k:18s} {round(v, 4) if isinstance(v, float) else v}")

best CV MAE : 0.706 Lakhs
best params :
   colsample_bytree   0.7637
   learning_rate      0.0382
   max_depth          7
   min_child_weight   1
   n_estimators       759
   reg_lambda         1.835
   subsample          0.7021


### Let the data pick `n_estimators`: **early stopping**
Instead of guessing the number of trees, give XGBoost far more than enough, hold
out a small validation slice of the *training* data, and stop as soon as the
validation error stops improving for `early_stopping_rounds` rounds. This is the
clean way to settle the learning-rate ↔ tree-count pair.

In [7]:
# Carve a validation slice out of the (already encoded) training set.
Xfit, Xval, yfit, yval = train_test_split(
    Xtr_e, ytr, test_size=0.2, random_state=RANDOM_STATE)

tuned = XGBRegressor(
    n_estimators=3000,          # generous upper bound; early stopping trims it
    learning_rate=0.03,         # small LR + many rounds => a finer, smoother fit
    max_depth=6, subsample=0.9, colsample_bytree=0.9,
    min_child_weight=3, reg_lambda=1.0,
    early_stopping_rounds=50, eval_metric="mae",
    tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, verbosity=0,
)
tuned.fit(Xfit, yfit, eval_set=[(Xval, yval)], verbose=False)

print("early stopping chose n_estimators =", tuned.best_iteration + 1)
print("baseline TEST :", score(yte, pred))
print("tuned    TEST :", score(yte, tuned.predict(Xte_e)))

early stopping chose n_estimators = 802
baseline TEST : {'R2': 0.9574, 'MAE': 0.665, 'RMSE': 1.014, 'band_acc': 0.8613}
tuned    TEST : {'R2': 0.9558, 'MAE': 0.676, 'RMSE': 1.034, 'band_acc': 0.8613}


### A practical tuning recipe
1. **Start sane:** `learning_rate=0.1`, `max_depth=6`, `subsample=0.9`,
   `colsample_bytree=0.9`.
2. **Set `n_estimators` by early stopping** (above), never by hand.
3. **Tune complexity:** search `max_depth` (3–10) and `min_child_weight` (1–10)
   *together* — they jointly control overfitting.
4. **Add randomness:** tune `subsample` and `colsample_bytree` in 0.6–1.0.
5. **Add regularisation** (`reg_lambda`, `reg_alpha`, `gamma`) only if you still
   overfit.
6. **Lower the learning rate** for the final model (0.1 → 0.03) and re-run early
   stopping — a small, reliable gain for longer training.

> **Why the project ships an *un*-tuned XGBoost:** the bake-off (nb 05) fixes
> modest, comparable settings across every model so the comparison is fair — the
> goal there is *selection*, not squeezing the last 0.1%. The recipe above is
> what you'd run *after* selection to productionise a chosen model.